In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sb
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.optimizers import Adam, RMSprop
from scikeras.wrappers import KerasRegressor
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.metrics import make_scorer, get_scorer

In [2]:
df = pd.read_csv(r"..\..\oblig3_og_4\abalone.data", sep=",", names=["Sex", "Length", "Diameter", "Height", "Whole weight", "Shucked weight", "Viscera weight", "Shell weight", "Rings"])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4177 entries, 0 to 4176
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Sex             4177 non-null   object 
 1   Length          4177 non-null   float64
 2   Diameter        4177 non-null   float64
 3   Height          4177 non-null   float64
 4   Whole weight    4177 non-null   float64
 5   Shucked weight  4177 non-null   float64
 6   Viscera weight  4177 non-null   float64
 7   Shell weight    4177 non-null   float64
 8   Rings           4177 non-null   int64  
dtypes: float64(7), int64(1), object(1)
memory usage: 293.8+ KB


In [3]:
df_targets = df['Rings']
features = df.drop('Rings', axis=1)
features['Sex'] = pd.Series(np.array(LabelEncoder().fit_transform(features['Sex'])))
df_features = pd.DataFrame(StandardScaler().fit_transform(features), columns=features.columns)

In [4]:
global_random_state = 15

scoring = {
    'mean_absolute_error': make_scorer(mean_absolute_error, greater_is_better=False),
    'mean_squared_error': make_scorer(mean_squared_error, greater_is_better=False),
    'r2': get_scorer('r2'),
}  

def evaluate(estimator, X, y):
    scores = {}
    for (name,scorer) in scoring.items():
        scores[name] = scorer(estimator, X, y) 
    return scores

def train(features, targets, estimator, params, scoring=scoring, refit='r2', random_state=global_random_state, outer_splits=5, inner_splits=4):

    outer_cv = StratifiedKFold(n_splits=outer_splits, shuffle=True, random_state=global_random_state)
    inner_cv = StratifiedKFold(n_splits=inner_splits, shuffle=True, random_state=global_random_state)

    scores_train = []
    scores_test = []
    estimators = []
    cv_results = []

    for (train_index, test_index) in outer_cv.split(features, targets):
        
        grid = GridSearchCV(
            estimator,
            params, 
            scoring=scoring, 
            refit=refit,
            error_score='raise', 
            cv=inner_cv)
        grid.fit(features.iloc[train_index], targets.iloc[train_index])        
        
        evaluation_train = evaluate(grid, features.iloc[train_index], targets.iloc[train_index])
        evaluation_test = evaluate(grid, features.iloc[test_index], targets.iloc[test_index])
        
        scores_train.append(evaluation_train)
        scores_test.append(evaluation_test)
        
        estimators.append(grid.best_estimator_)
        cv_results.append(pd.DataFrame(grid.cv_results_))
        print("*")

    return estimators, pd.DataFrame(scores_train), pd.DataFrame(scores_test), pd.concat(cv_results, names=['test_split'], keys=range(outer_splits))

In [5]:
rf_params = {
    'max_depth': [10],
    'min_samples_leaf': [8],
    'criterion': ['squared_error'],
    'n_estimators': [1000],
}

rf_estimators_final, rf_scores_train_final, rf_scores_test_final, rf_cv_results_final = train(df_features, df_targets,
    RandomForestRegressor(random_state=global_random_state),
    rf_params, outer_splits=5, inner_splits=4)

c:\Users\hallo\Documents\GitHub\praktisk-maskinlering\.venv\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\hallo\Documents\GitHub\praktisk-maskinlering\.venv\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=4.
  warnings.warn(


*


c:\Users\hallo\Documents\GitHub\praktisk-maskinlering\.venv\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=4.
  warnings.warn(


*


c:\Users\hallo\Documents\GitHub\praktisk-maskinlering\.venv\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=4.
  warnings.warn(


*


c:\Users\hallo\Documents\GitHub\praktisk-maskinlering\.venv\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=4.
  warnings.warn(


*


c:\Users\hallo\Documents\GitHub\praktisk-maskinlering\.venv\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=4.
  warnings.warn(


*


In [6]:
def create_model(optimizer='adam', activation='relu', hidden_neurons=75, layers=2, reduction_factor=0.5, dropout=0, learning_rate=0.001):
    model = Sequential()
    model.add(Input(shape=(8,)))
    model.add(Dense(hidden_neurons, activation=activation))
    for layer in range(0, layers+1):
        hidden_neurons = int(hidden_neurons*reduction_factor)
        if(hidden_neurons >= 1):
            model.add(Dense(hidden_neurons, activation=activation))
    if(dropout > 0):
        model.add(Dropout(dropout))
    model.add(Dense(1))
    if(optimizer == 'adam'):
        optimizer = Adam(learning_rate=learning_rate)
    else:
        optimizer = RMSprop(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='mean_squared_error', metrics=['mean_squared_error'])
    return model

keras_sequential_1 = KerasRegressor(model=create_model, verbose=0)

keras_reg_param = {
    'batch_size': [10],
    'epochs': [40],
    'model__hidden_neurons': [150],
    'model__layers': [2],
    'model__reduction_factor': [0.5],
    'model__optimizer': ['rmsprop'],
    'model__activation': ['elu'],
    'model__learning_rate': [0.001],
    'model__dropout':[0],
}
keras_sequential_estimators_final, keras_sequential_scores_train_final, keras_sequential_scores_test_final, keras_sequential_cv_results_final = train(df_features, df_targets, keras_sequential_1, keras_reg_param, outer_splits=5, inner_splits=4)

c:\Users\hallo\Documents\GitHub\praktisk-maskinlering\.venv\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\hallo\Documents\GitHub\praktisk-maskinlering\.venv\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=4.
  warnings.warn(


*


c:\Users\hallo\Documents\GitHub\praktisk-maskinlering\.venv\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=4.
  warnings.warn(


*


c:\Users\hallo\Documents\GitHub\praktisk-maskinlering\.venv\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=4.
  warnings.warn(


*


c:\Users\hallo\Documents\GitHub\praktisk-maskinlering\.venv\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=4.
  warnings.warn(


*


c:\Users\hallo\Documents\GitHub\praktisk-maskinlering\.venv\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=4.
  warnings.warn(


*


In [7]:
slutt_resultat = pd.DataFrame({
    'Metric': ['Mean absolute error:','Mean squared error:','r2:'],
    'RandomForest Train': [rf_scores_train_final["mean_absolute_error"].mean(), rf_scores_train_final["mean_squared_error"].mean(), rf_scores_train_final['r2'].mean()],
    'RandomForest Test': [rf_scores_test_final["mean_absolute_error"].mean(), rf_scores_test_final["mean_squared_error"].mean(), rf_scores_test_final['r2'].mean()],
    'Keras Sequential Train': [keras_sequential_scores_train_final["mean_absolute_error"].mean(), keras_sequential_scores_train_final["mean_squared_error"].mean(), keras_sequential_scores_train_final['r2'].mean()],
    'Keras Sequential Test': [keras_sequential_scores_test_final["mean_absolute_error"].mean(), keras_sequential_scores_test_final["mean_squared_error"].mean(), keras_sequential_scores_test_final['r2'].mean()],
})
slutt_resultat.set_index('Metric', inplace=True)
slutt_resultat

,RandomForest Train,RandomForest Test,Keras Sequential Train,Keras Sequential Test
Metric,,,,
Mean absolute error:,-1.207375,-1.501076,-1.449595,-1.515237
Mean squared error:,-2.971333,-4.561270,-4.138688,-4.619258
r2:,0.714112,0.561307,0.601818,0.555376


### Sammenligning av RandomForestRegressor og Keras Sequetial

- Som vi ser ut i fra resultatene over så gir RandomForest best resultat for alle metricene og modellen gir henholdsvis en prestasjon på r2: 0.561, mean absolute error: 1.501, mean squared error: 4.561 for RandomForest og r2: 0.555, mean absolute error: 1.515, mean squared error: 4.619 for Keras Sequential.

- Hvis vi sammenligner forskjellene mellom train og test for modellene ser vi at RandomForst har større forskjell mellom test og train enn for Keras Sequential. Dette kan tyde på at RandomForst modellen har noe problemere med overfit og er noe som er vært å ta med i en helhetlig vurdering.

- For Keras Sequential og Keras modellene generelt for abalone regresjons datasettet har jeg hatt problmere med uteligger verdier for modelen, dette i kombinasjon med at jeg har glemt å legge inn et fast randomseed for neural nettwork modellene har gjort at reultatene for modellen er ustabile ved gjengatagende treninger/kjøringer av modellen. I utgangspunktet har Keras Sequential gitt bedre resultater nå jeg sammenlignet denne med den andre Keras modellen for abalone datasettet. For den analysen fikk jeg test resultater på r2: 0.571, mean absolute error: 1,513 og mean squeard error: 0,463, men når jeg trener modellen flere gang endrer disse resultatene seg betydelig og kan i flere tilfeller blir betydelig lavere.

- Basert på disse resultatene og en helhetlig vurdering ville jeg ha valg RandomForest modellen som den beste modellen men det er vært å nevne at ved videre tuning og en bedre arkitektur for neural nettwork modellene er det mulig at disse vil prestere bedre og mer stabilt enn det jeg har fått til. 

- Helhetlig betyr dette at  RandomForest modellene har en god overførings verdi til å kunne gi en grov estimering av alder/antall ringer for abalonene, men at den ikke egner seg til å gi presise prediksjonere. I kilden til datasette kommenteres det at det er behov for mer data, samt at det potensielt er nødvendig med features som beskrive ytre miljø påvirkninger for abalonene for å kunne utvikle modeller med høyere korellasjo som dermed kan gi mer presise prediksjoner for alder/antall ringer [1].

[1]: UC Irvine. (u.å) Abalone. Hentet 9. November 2025 fra https://archive.ics.uci.edu/dataset/1/abalone

- Ved sammenligning av modellen før endelig mappe innlevering har jeg oppdaget at jeg dessverre ikke har lagt inn fast random seed for selve modellene for neural nettwork modellen, det er bruk fast random seed for oppdelingen av datasettet i nestedCV, men ikke for selve modellene slik det har blir gjort for supervised modellene. Dette gjør dessverre krevende å sammenligne resultatene mellom modellen, men jeg har dessverre ikke hatt tid til å rett opp i denne feilen før innleveringen av den endelige mappem. 